In [5]:
# Pega os lembretes novos na planilha

import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

scope = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]

creds = ServiceAccountCredentials.from_json_keyfile_name('secret.json', scope)
client = gspread.authorize(creds)

spreadsheet = client.open_by_key("1gjz0hk8dL0rbqzTfHGLbzwXnA-h8AVxZtgffCbseDIk")
sheet = spreadsheet.sheet1

data = sheet.get_all_records()
df = pd.DataFrame(data)
df = df.loc[:, (df != '').any(axis=0)]

df.rename(columns={"Tipo de Pedido": "tipo_pedido"}, inplace=True)
df["num_processo"] = df["num_processo"].str.strip()

In [6]:
df


,unidade,num_processo,tipo_pedido
0,STM1CIV,5045054-21.2024.8.21.0027,extinção 2
1,STM1CIV,5044977-12.2024.8.21.0027,extinção 2
2,STM1CIV,5042475-37.2023.8.21.0027,extinção 2
3,STM1CIV,5042347-17.2023.8.21.0027,extinção 2
4,STM1CIV,5041448-53.2022.8.21.0027,extinção 2
...,...,...,...
126,SRD1CIV,5000152-71.2012.8.21.0069,
127,SRD1CIV,5000126-15.2008.8.21.0069,
128,SRD1CIV,5000122-79.2025.8.21.0069,
129,SRD1CIV,5000087-22.2025.8.21.0069,


In [7]:
# Reseta a tabela

import sqlite3
db = "urcaciv.db"
conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"DELETE FROM lembretes_to_do")
conn.commit()

conn.close()

print("Todos os registros da tabela 'lembretes_to_do' foram apagados.")

Todos os registros da tabela 'lembretes_to_do' foram apagados.


In [8]:
from datetime import datetime

df["created_at"] = datetime.now().strftime("%d/%m/%Y")
df["dt_lembrete"] = None

print(df)
df.to_sql("lembretes_to_do", "sqlite:///urcaciv.db", if_exists="append", index=False)

     unidade               num_processo tipo_pedido  created_at dt_lembrete
0    STM1CIV  5045054-21.2024.8.21.0027  extinção 2  23/09/2025        None
1    STM1CIV  5044977-12.2024.8.21.0027  extinção 2  23/09/2025        None
2    STM1CIV  5042475-37.2023.8.21.0027  extinção 2  23/09/2025        None
3    STM1CIV  5042347-17.2023.8.21.0027  extinção 2  23/09/2025        None
4    STM1CIV  5041448-53.2022.8.21.0027  extinção 2  23/09/2025        None
..       ...                        ...         ...         ...         ...
126  SRD1CIV  5000152-71.2012.8.21.0069              23/09/2025        None
127  SRD1CIV  5000126-15.2008.8.21.0069              23/09/2025        None
128  SRD1CIV  5000122-79.2025.8.21.0069              23/09/2025        None
129  SRD1CIV  5000087-22.2025.8.21.0069              23/09/2025        None
130  SRD1CIV  5000058-55.2014.8.21.0069              23/09/2025        None

[131 rows x 5 columns]


131